# 데이터 전처리

In [2]:
import pandas as pd
import numpy as np
import requests
import time
import re
import os
import sys
import io
from tqdm import tqdm
import warnings
import math
import unicodedata
from difflib import SequenceMatcher
from typing import Optional, Any
import requests
import json
import streamlit as st
from datetime import datetime, timedelta
from dotenv import load_dotenv
from functools import lru_cache
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

c:\Users\Keon\anaconda3\envs\project_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 위경도 보정

KAKAO_API_KEY = ""
INPUT_FILE = "서울 숙박 리스트(한국관광콘텐츠랩)_최종.csv"
OUTPUT_FILE = "서울_숙박_위경도_보정_리스트.csv"

# 서울 위경도 범위 지정
LAT_RANGE = (37.2, 37.9)
LON_RANGE = (126.5, 127.4)

def get_kakao_coords(query, is_address=True):
    endpoint = "address.json" if is_address else "keyword.json"
    url = f"https://dapi.kakao.com/v2/local/search/{endpoint}"
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {"query": query}
    
    try:
        response = requests.get(url, headers=headers, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['documents']:
                res = data['documents'][0]
                return {
                    "lat": res.get('y'), # 위도
                    "lon": res.get('x')  # 경도
                }
        return None
    except: return None

def main():
    df = None
    for enc in ['utf-8-sig', 'cp949', 'euc-kr']:
        try:
            df = pd.read_csv(INPUT_FILE, encoding=enc)
            break
        except: continue
    
    if df is None:
        print("❌ 파일을 찾을 수 없습니다.")
        return

    strange_mask = (
        df['위도'].isna() | df['경도'].isna() |
        (df['위도'] == 0) | (df['경도'] == 0) |
        (df['위도'] < LAT_RANGE[0]) | (df['위도'] > LAT_RANGE[1]) |
        (df['경도'] < LON_RANGE[0]) | (df['경도'] > LON_RANGE[1])
    )
    
    targets = df[strange_mask].copy()
    print(f"🔎 보정 대상: {len(targets)}건 발견")

    corrected_data = []
    for idx, row in targets.iterrows():
        place_name = row['명칭']
        address = row['주소'] if not pd.isna(row['주소']) else row.get('ADDR1', "")
        
        res = None
        # 주소로 검색 시도
        if address:
            print(f"🔄 [1차] 주소 검색 중: {place_name}")
            res = get_kakao_coords(address, is_address=True)
        
        # 주소 검색 실패 시 장소명으로 재검색
        if not res:
            print(f"🔄 [2차] 장소명 검색 중: {place_name}")
            res = get_kakao_coords(place_name, is_address=False)
        
        if res:
            corrected_data.append({
                "장소명": place_name,
                "위도": res['lat'],
                "경도": res['lon']
            })
            print(f"   ✅ 성공")
        else:
            print(f"   ❌ 실패 (정보 부족)")
        
        time.sleep(0.1)

    if corrected_data:
        pd.DataFrame(corrected_data).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
        print(f"\n✨ 완료! 보정된 리스트가 '{OUTPUT_FILE}'에 저장되었습니다. (총 {len(corrected_data)}건)")

if __name__ == "__main__":
    main()

🔎 보정 대상: 1건 발견
🔄 [1차] 주소 검색 중: 롯데시티호텔 김포공항
   ✅ 성공

✨ 완료! 보정된 리스트가 '서울_숙박_위경도_보정_리스트.csv'에 저장되었습니다. (총 1건)


In [ ]:
# 구, 법정동 역지오코딩

KAKAO_REST_API_KEY = ""

def get_legal_dong(lat, lng, api_key):
    url = f"https://dapi.kakao.com/v2/local/geo/coord2regioncode.json?x={lng}&y={lat}"
    headers = {"Authorization": f"KakaoAK {api_key}"}
    
    try:
        response = requests.get(url, headers=headers, timeout=5)
        if response.status_code == 200:
            data = response.json()
            for region in data.get('documents', []):
                if region['region_type'] == 'B':
                    return region['region_2depth_name'], region['region_3depth_name']
    except Exception as e:
        print(f"오류 발생: {e}")
    
    return None, None

df = pd.read_csv('서울 카페 리스트_최종.csv')

gu_results = []
dong_results = []

print("🚀 카카오 API로 법정동 수집중...")

for i, row in tqdm(df.iterrows(), total=len(df)):
    gu, dong = get_legal_dong(row['위도'], row['경도'], KAKAO_REST_API_KEY)
    gu_results.append(gu)
    dong_results.append(dong)
    
    time.sleep(0.05)

df['구'] = gu_results
df['법정동'] = dong_results

df.to_csv('서울 카페 리스트_구, 법정동 파싱까지.csv', index=False, encoding='utf-8-sig')
print("\n✅ 완료! '서울 카페 리스트_구, 법정동 파싱까지.csv' 파일이 생성되었습니다.")

🚀 카카오 API로 법정동 수집중...


100%|██████████| 2500/2500 [04:44<00:00,  8.78it/s]


✅ 완료! '서울 카페 리스트_구, 법정동 파싱까지.csv' 파일이 생성되었습니다.


In [ ]:
# 지하철역 데이터 전처리 - 역명 끝에 '역' 텍스트가 없다면 붙여주기

df3 = pd.read_csv('서울 지하철 정보.csv', encoding='cp949', low_memory=False)

def add_station_suffix(name):
    name = str(name).strip()
    # 괄호가 있는 경우 괄호 앞에 있는 텍스트 마지막에 붙이기
    if '(' in name:
        match = re.match(r'([^(]+)(\(.*\))', name)
        if match:
            main_name = match.group(1)
            sub_name = match.group(2)
            if not main_name.endswith('역'):
                return f"{main_name}역{sub_name}"
            return name
    
    if not name.endswith('역'):
        return name + '역'
    
    return name

df3['역명'] = df3['역명'].apply(add_station_suffix)

df3.to_csv('서울 지하철 정보_역명수정.csv', index=False, encoding='utf-8-sig')

In [3]:
# 주소 키워드 파생변수 - 구, 대표 법정동, 세부 법정동

df = pd.read_csv('서울 카페 리스트_구, 법정동 파싱까지.csv', encoding='utf-8-sig', low_memory=False)

# 대표 법정동 파싱 함수
def get_representative_dong(dong):
    if pd.isna(dong):
        return ""
    dong = str(dong)
    # 숫자+가 제거
    dong = re.sub(r'\d+가$', '', dong)
    # 숫자+동에서 숫자만 제거
    dong = re.sub(r'(\d+)동$', '동', dong)
    return dong

df['주소 키워드'] = df.apply(
    lambda row: f"{row['구명']}, {get_representative_dong(row['법정동명'])}, {row['법정동명'] if pd.notna(row['법정동명']) else ''}", 
    axis=1
)

df.to_csv('서울 카페 리스트_최종.csv', encoding='utf-8-sig', index=False)

In [105]:
# 역 키워드 파생 변수(근처 1km 이내 역)

# 데이터 로드
subway_df = pd.read_csv("서울 지하철 정보_역명 수정.csv", encoding="utf-8-sig")

# 하버사인 거리 계산 함수 (단위: km)
def haversine_distance(lat1, lon1, lat2, lon2):
    # 위경도를 라디안 단위로 변환
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    # 하버사인 공식
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371 # 지구의 반지름 (km)
    
    return c * r

keywords = []

# 각 장소별로 1km 이내 모든 지하철역 탐색
for idx, row in df.iterrows():
    t_lat = row['위도']
    t_lon = row['경도']
    
    # 만약 위도/경도 값이 비어있다면 처리 패스
    if pd.isna(t_lat) or pd.isna(t_lon):
        keywords.append("정보없음")
        continue

    # 현재 장소와 '모든 지하철역'들 사이의 거리를 한 번에 계산
    dists = haversine_distance(t_lat, t_lon, subway_df['위도'].values, subway_df['경도'].values)
    
    # 거리가 1.0km 이하인 역들의 '인덱스' 모두 찾기
    within_1km_idx = np.where(dists <= 1.0)[0]
    
    if len(within_1km_idx) > 0:
        # 1km 이내 역들의 이름과 거리를 배열로 추출
        nearby_stations = subway_df.iloc[within_1km_idx]['역명'].values
        nearby_dists = dists[within_1km_idx]
        
        # 거리가 가까운 순서대로 정렬 (가장 가까운 역이 맨 앞에 오도록)
        sorted_indices = np.argsort(nearby_dists)
        sorted_stations = nearby_stations[sorted_indices]
        
        unique_clean_stations = []
        for stat in sorted_stations:
            # "교대역(법원.검찰청)"처럼 괄호가 있으면 제거하고 깔끔하게 이름만 추출
            clean_stat = str(stat).split('(')[0].strip()
            
            # 환승역 중복 방지 (이미 들어간 역명이면 무시)
            if clean_stat not in unique_clean_stations:
                unique_clean_stations.append(clean_stat)
                
        keyword_str = ", ".join(unique_clean_stations)
        keywords.append(keyword_str)
        
    else:
        # 1km 이내에 역이 하나도 없는 경우
        keywords.append("근처역 없음")

df['근처역 키워드'] = keywords

In [107]:
# # 필요없는 컬럼 제거(관광)
# df = df.drop(['우편번호', '관리자', '전화번호', '유산구분', '문의 및 안내', '개장일', '쉬는날', '체험가능연령', '수용인원', '이용시기', '이용시간', '애완동물 동반 가능 여부',
#               '신용카드 가능 여부', '구명', '법정동명'], axis=1)

# # 필요없는 컬럼 제거(쇼핑)
# df = df.drop(['우편번호', '관리자', '전화번호', '문의 및 안내', '규모', '품목별 가격', '장서는 날', '개장일', '영업시간', '쉬는날', '매장안내', '문화센터 바로가기', '애완동물 동반 가능 여부',
#               '구명', '법정동명'], axis=1)

# 필요없는 컬럼 제거(숙박)
df = df.drop(['우편번호', '관리자', '전화번호', '숙박 종류', '문의 및 안내', '규모', '수용 가능 인원', '객실 수', '체크인', '체크아웃', '예약 안내', '예약안내 홈페이지', 
              '세미나', '스포츠시설', '사우나실', '뷰티 시설', '노래방', '바베큐장', '캠프화이어', '자전거대여', '휘트니스센터', '공용 PC실', '공용 샤워실', '상세정보', 
              '환불규정', '구명', '법정동명'], axis=1)

In [109]:
df = df.rename(columns={'전화번호.1' : '전화번호', '체크인.1' : '체크인', '체크아웃.1' : '체크아웃'})

In [111]:
# # 컬럼 순서 변경(관광)
# df = df[['대분류', '소분류', '명칭', '주소', '주소 키워드', '위도', '경도', '근처역 키워드', '개요', '전화번호', '홈페이지', '운영시간', '휴무일', '이용요금', '체험안내', '주차시설', 
#          '유모차 대여 여부', '상세정보', '총 평점', '리뷰 총 개수']]

# # 컬럼 순서 변경(쇼핑)
# df = df[['대분류', '소분류', '명칭', '주소', '주소 키워드', '위도', '경도', '근처역 키워드', '개요', '전화번호', '홈페이지', '운영시간', '휴무일', '판매품목', '주차시설', 
#          '유모차 대여 여부', '화장실', '신용카드 가능 여부', '기타', '총 평점', '리뷰 총 개수']]

# 컬럼 순서 변경(쇼핑)
df = df[['대분류', '소분류', '명칭', '호텔 성급', '주소', '주소 키워드', '위도', '경도', '근처역 키워드', '개요', '전화번호', '홈페이지', '객실 유형', '체크인', '체크아웃', 
         '조리 가능', '픽업서비스', '주차시설', '식음료장', '부대 시설', '총 평점', '리뷰 총 개수']]

In [52]:
# 데이터 로드
df = pd.read_csv("확인.csv", encoding="utf-8-sig")

def sophisticated_parser(text):
    if pd.isna(text):
        return None, None, None
    
    # 추출할 키워드 정의 (정규식 대응: 띄어쓰기 무시)
    # 요금 관련: 입장료, 시설이용료, 이용요금
    # 화장실 관련: 화장실
    fee_keywords = ['입\s*장\s*료', '시\s*설\s*이\s*용\s*료', '이\s*용\s*요\s*금']
    toilet_keyword = '화\s*장\s*실'
    
    # 모든 키워드를 하나로 합침 (구분자로 사용)
    all_keywords = fee_keywords + [toilet_keyword]
    # 기타 다른 가능한 키워드들도 추가 (경계선을 더 정확히 잡기 위함)
    boundary_keywords = all_keywords + ['체험\s*프로그램', '관광\s*코스\s*안내', '등산로', '이용\s*가능\s*시설', '촬영\s*장소', '예약\s*안내']
    
    pattern_combined = '|'.join(boundary_keywords)
    
    extracted_fee = []
    extracted_toilet = []
    remaining_text = str(text)
    
    # 1단계: 요금 관련 정보 추출
    for kw in fee_keywords:
        # 키워드: 부터 다음 키워드: 전까지 혹은 끝까지 추출
        regex = rf"({kw}\s*:.*?)(?=(?:<br>|\n)\s*(?:{pattern_combined})\s*:|$)"
        matches = re.finditer(regex, remaining_text, flags=re.IGNORECASE | re.DOTALL)
        for m in matches:
            info = m.group(1).strip()
            extracted_fee.append(info)
            remaining_text = remaining_text.replace(m.group(1), "") # 원본에서 삭제
            
    # 2단계: 화장실 정보 추출
    regex_toilet = rf"({toilet_keyword}\s*:.*?)(?=(?:<br>|\n)\s*(?:{pattern_combined})\s*:|$)"
    matches_toilet = re.finditer(regex_toilet, remaining_text, flags=re.IGNORECASE | re.DOTALL)
    for m in matches_toilet:
        info = m.group(1).strip()
        extracted_toilet.append(info)
        remaining_text = remaining_text.replace(m.group(1), "") # 원본에서 삭제

    # 결과 정리
    fee_res = " / ".join(extracted_fee) if extracted_fee else None
    toilet_res = " / ".join(extracted_toilet) if extracted_toilet else None
    
    # 남은 텍스트 정리 (불필요한 공백 및 줄바꿈 제거)
    remaining_text = re.sub(r'\n+', '\n', remaining_text).strip()
    remaining_text = re.sub(r'^\s*<br>\s*', '', remaining_text)
    
    return fee_res, toilet_res, remaining_text

# 2. 파싱 적용
parsed_results = df['기타'].apply(sophisticated_parser)
df['추출요금'], df['화장실'], df['기타_정리'] = zip(*parsed_results)

# 3. 로직 적용: 기존 이용요금이 결측치(NaN/공백)일 때만 추출된 요금 삽입
# 기존 이용요금의 공백을 NaN으로 변경
df['이용요금'] = df['이용요금'].replace(r'^\s*$', np.nan, regex=True)
df['이용요금'] = df['이용요금'].fillna(df['추출요금'])

# 4. 마무리 작업
# '기타' 컬럼을 정리된 텍스트로 덮어쓰기
df['기타'] = df['기타_정리']

# 임시 컬럼 삭제 및 최종 빈칸 '정보없음' 처리
df = df.drop(columns=['추출요금', '기타_정리'])
final_cols = ['이용요금', '화장실', '기타']
for col in final_cols:
    df[col] = df[col].fillna('정보없음')

# 5. 저장
df.to_csv("서울_관광_전처리_완료.csv", index=False, encoding="utf-8-sig")

In [3]:
# 장소 통합 Chroma DB 생성
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cuda'}, 
    encode_kwargs={'normalize_embeddings': True}
)

vector_store = Chroma(persist_directory="./seoulmate_place_db", embedding_function=embeddings)

In [4]:
files_to_process = [
    {"file": "서울 관광 리스트_최종.csv", "theme": "관광"},
    {"file": "서울 쇼핑 리스트_최종.csv", "theme": "쇼핑"},
    {"file": "서울 숙박 리스트_최종.csv", "theme": "숙박"},
    {"file": "서울 음식 리스트_최종.csv", "theme": "음식"},
    {"file": "서울 카페 리스트_최종.csv", "theme": "카페"}
]

# 숫자형 데이터 안전 변환
def safe_float(val):
    try: return float(val) if val != "" else 0.0
    except: return 0.0

# 문자형 결측치 안전 변환 함수 (핵심)
def safe_str(val):
    if pd.isna(val) or str(val).strip().lower() in ['nan', 'none', '정보 없음', '정보없음', '']:
        return ""
    return str(val).strip()

documents = []
print("데이터 로딩 및 문서 통합 작업 시작...")

for info in files_to_process:
    theme = info["theme"]
    try:
        df = pd.read_csv(info["file"], encoding="utf-8-sig")
        df = df.fillna("")

        for idx, row in df.iterrows():
            if theme == "숙박":
                checkin = safe_str(row.get('체크인', ''))
                checkout = safe_str(row.get('체크아웃', ''))
                time_info = f"체크인 {checkin} / 체크아웃 {checkout}" if checkin and checkout else ""
            else:    
                operating_hours = safe_str(row.get('운영시간', row.get('이용시간', row.get('영업시간', ''))))
                closed_days = safe_str(row.get('휴무일', ''))
                
                if closed_days and closed_days not in ['없음', '연중무휴', '정보 없음', '정보없음']:
                    time_info = f"{operating_hours} (휴무일: {closed_days})"
                else:
                    time_info = operating_hours

            description = safe_str(row.get('개요', row.get('장소 키워드', '')))
            page_content = (
                f"테마: {theme}\n"
                f"명칭: {row['명칭']}\n"
                f"주소: {row.get('주소', '')}\n"
                f"시간: {time_info}\n"
                f"설명: {description}"
            )
            
            # 통합 메타데이터 스키마
            metadata = {
                "theme": theme,
                "name": safe_str(row.get('명칭', '')),
                "main_category": safe_str(row.get('대분류', '')),
                "sub_category": safe_str(row.get('소분류', '')),
                "address": safe_str(row.get('주소', '')),
                "gu_dong": safe_str(row.get('주소 키워드', '')),
                "station": safe_str(row.get('근처역 키워드', '')),
                "rating": safe_float(row.get('총 평점', 0)),
                "review_cnt": int(safe_float(row.get('리뷰 총 개수', 0))),
                "lat": safe_float(row.get('위도', 0)),
                "lng": safe_float(row.get('경도', 0)),
                "keywords": safe_str(row.get('장소 키워드', '')),
                "phone": safe_str(row.get('전화번호', row.get('연락처', ''))),
                "homepage": safe_str(row.get('홈페이지', '')),
                "time_info": time_info,
                "parking": safe_str(row.get('주차시설', '')),
                "misc": safe_str(row.get('기타', '')),
                # [관광]
                "fee": safe_str(row.get('이용요금', '')),
                "program": safe_str(row.get('체험 프로그램 안내', '')),
                # [쇼핑]
                "items": safe_str(row.get('판매품목', '')),
                # [숙박]
                "room_type": safe_str(row.get('객실 유형', '')),
                "star_rating": safe_str(row.get('호텔 성급', '')),
                "cooking": safe_str(row.get('조리 가능', '')),
                "pickup": safe_str(row.get('픽업서비스', '')),
                "fnb": safe_str(row.get('식음료장', '')),
                "facilities": safe_str(row.get('부대 시설', ''))
            }
            
            doc = Document(page_content=page_content, metadata=metadata)
            documents.append(doc)
            
        print(f"✅ [{theme}] 데이터 {len(df)}건 처리 완료")
    except Exception as e:
        print(f"❌ [{theme}] 파일 처리 중 에러 발생: {e}")

print("통합 벡터 DB 구축 및 저장을 시작합니다...")
vector_store = Chroma.from_documents(documents=documents, embedding=embeddings, persist_directory="./seoulmate_place_db")
print("\n🎉 성공적으로 DB가 구축되었습니다!")

데이터 로딩 및 문서 통합 작업 시작...
✅ [관광] 데이터 1134건 처리 완료
✅ [쇼핑] 데이터 236건 처리 완료
✅ [숙박] 데이터 299건 처리 완료
✅ [음식] 데이터 6158건 처리 완료
✅ [카페] 데이터 2500건 처리 완료
통합 벡터 DB 구축 및 저장을 시작합니다...

🎉 성공적으로 DB가 구축되었습니다!


In [4]:
# 리뷰 통합 Chroma DB 생성
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cuda'}, 
    encode_kwargs={'normalize_embeddings': True}
)

vector_store = Chroma(persist_directory="./seoulmate_review_db", embedding_function=embeddings)

In [5]:
# 리뷰 임베딩 및 db화
def build_review_db(file_path, theme_name):
    try:
        # 파일 읽기 및 내용 없는 리뷰 제거
        df = pd.read_csv(file_path, encoding='utf-8-sig')
        df = df.dropna(subset=['내용'])
        
        # 장소명이나 주소가 비어있어서 그룹화가 터지는 현상 방지
        df['원본장소명'] = df['원본장소명'].fillna('이름 없음')
        df['원본주소'] = df['원본주소'].fillna('주소 없음')

        all_docs = []
        print(f"\n🚀 {theme_name} 리뷰 가공 시작...")

        # '원본장소명'과 '원본주소'를 동시에 기준으로 그룹화하여 지점 구분
        grouped = df.groupby(['원본장소명', '원본주소'])
        
        for (name, address), group in tqdm(grouped):
            reviews = group['내용'].tolist()
            
            # 리뷰 10개씩 묶어서 저장
            for i in range(0, len(reviews), 10):
                chunk = reviews[i:i+10]
                
                # 검색 품질을 위해 본문에 이름과 주소를 포함
                combined_text = f"장소명: {name} | 위치: {address} | 실제 방문객 리뷰: {' '.join(chunk)}"
                
                all_docs.append(Document(
                    page_content=combined_text,
                    metadata={
                        "name": str(name),
                        "address": str(address),
                        "theme": theme_name,
                        "type": "review" 
                    }
                ))

        # DB에 추가 저장 (500개씩 배치 처리)
        if all_docs:
            print(f"💾 {theme_name} 데이터 DB 추가 중 (총 {len(all_docs)}개 뭉치)...")
            for i in range(0, len(all_docs), 500):
                batch = all_docs[i:i+500]
                vector_store.add_documents(batch)
            print(f"✅ {theme_name} 저장 완료!")
        else:
            print(f"⚠️ {theme_name} 리뷰 데이터가 없습니다.")
            
    except Exception as e:
        print(f"❌ {file_path} 처리 중 에러 발생: {e}")

# 모든 리뷰 데이터 파일 순차 실행
if __name__ == "__main__":
    build_review_db("서울 관광_구글 지도 리뷰_최종.csv", "관광")
    build_review_db("서울 쇼핑_구글 지도 리뷰_최종.csv", "쇼핑")
    build_review_db("서울 숙박_구글 지도 리뷰_최종.csv", "숙박")
    
    print("\n🎉 모든 리뷰 데이터가 성공적으로 리뷰 전용 DB에 추가되었습니다!")


🚀 관광 리뷰 가공 시작...


100%|██████████| 1134/1134 [00:00<00:00, 12955.15it/s]


💾 관광 데이터 DB 추가 중 (총 8352개 뭉치)...
✅ 관광 저장 완료!

🚀 쇼핑 리뷰 가공 시작...


100%|██████████| 236/236 [00:00<00:00, 15372.10it/s]

💾 쇼핑 데이터 DB 추가 중 (총 2318개 뭉치)...


✅ 쇼핑 저장 완료!

🚀 숙박 리뷰 가공 시작...


100%|██████████| 299/299 [00:00<00:00, 6781.25it/s]


💾 숙박 데이터 DB 추가 중 (총 3028개 뭉치)...
✅ 숙박 저장 완료!

🎉 모든 리뷰 데이터가 성공적으로 리뷰 전용 DB에 추가되었습니다!
